In [1]:
import os
os.getcwd()

'd:\\Programming\\labwork\\ml'

In [2]:
import numpy as np
import matplotlib.pyplot as plt

In [3]:

def load_csv(filename):
    return np.loadtxt(filename, delimiter=',')

def kmeans(X, K, max_iters=100):
    np.random.seed(0)
    centroids = X[np.random.choice(X.shape[0], K, replace=False)]
    prev_labels = None
    for _ in range(max_iters):
        distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
        labels = np.argmin(distances, axis=1)
        new_centroids = np.zeros_like(centroids)
        for k in range(K):
            cluster_points = X[labels == k]
            if len(cluster_points) == 0:
                new_centroids[k] = X[np.random.choice(X.shape[0])]
            else:
                new_centroids[k] = cluster_points.mean(axis=0)
        if np.all(labels == prev_labels):
            break
        centroids = new_centroids
        prev_labels = labels
    return centroids, labels

def compute_wcss(X, centroids, labels):
    return np.sum(np.sum((X - centroids[labels]) ** 2, axis=1))

def elbow_kneedle(Kmax, wcss):
    x1, y1 = 1, wcss[0]
    x2, y2 = Kmax, wcss[-1]
    distances = []
    for K in range(1, Kmax + 1):
        num = abs((y2 - y1) * K - (x2 - x1) * wcss[K-1] + x2 * y1 - y2 * x1)
        den = np.sqrt((y2 - y1)**2 + (x2 - x1)**2)
        distances.append(num / den)
    return np.argmax(distances) + 1

def main():
    import sys
    lines = sys.stdin.read().strip().splitlines()
    Kmax = int(lines[0].split('=')[1])
    iterations = int(lines[1].split('=')[1])

    X_train = load_csv('k_means_train.csv')
    X_val = load_csv('k_means_val.csv')
    X_test = load_csv('k_means_test.csv')

    wcss_val = []
    for K in range(1, Kmax + 1):
        centroids, labels = kmeans(X_train, K, iterations)
        val_labels = np.argmin(np.linalg.norm(X_val[:, np.newaxis] - centroids, axis=2), axis=1)
        wcss_val.append(compute_wcss(X_val, centroids, val_labels))

    elbow_K = elbow_kneedle(Kmax, wcss_val)
    final_centroids, final_labels = kmeans(X_train, elbow_K, iterations)

    test_labels = np.argmin(np.linalg.norm(X_test[:, np.newaxis] - final_centroids, axis=2), axis=1)
    test_J = np.mean(np.sum((X_test - final_centroids[test_labels])**2, axis=1))
    test_WCSS = compute_wcss(X_test, final_centroids, test_labels)

    print(f"Elbow_K={elbow_K}")
    print(f"WCSS_Val={[round(v, 2) for v in wcss_val]}")
    print(f"Test_J_Elbow={test_J:.2f} Test_WCSS_Elbow={test_WCSS:.2f}")

    plt.plot(range(1, Kmax + 1), wcss_val, marker='o')
    plt.title("Elbow Method")
    plt.xlabel("K")
    plt.ylabel("WCSS (Validation)")
    plt.savefig("elbow_curve.png")
    plt.close()


In [ ]:
if __name__ == "__main__":
    main() 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Data based on the provided text ---
# Fiscal years
years = ['FY21', 'FY22', 'FY23', 'FY24', 'FY25 (Proj.)']

# FDI Inflow in Renewables (in US$ Billions).
# The total is ~$20 billion, distributed here with an upward trend.
fdi_inflow_usd_billion = [1.5, 3.5, 5.0, 6.0, 4.0] # Sums to 20

# Renewables' Share of Total FDI (in %).
# This shows the growth from ~1% to a projected ~8%.
share_of_total_fdi_percent = [1.0, 2.8, 4.5, 6.5, 8.0]


# --- Graph Creation ---
# Create a figure and a set of subplots
fig, ax1 = plt.subplots(figsize=(12, 7))

# Set the style for a more modern look
plt.style.use('seaborn-v0_8-whitegrid')

# --- Bar Chart (FDI Inflow) ---
# Define bar color and width
bar_color = 'skyblue'
bar_width = 0.6
bars = ax1.bar(years, fdi_inflow_usd_billion, color=bar_color, label='FDI Inflow (US$ Billions)', width=bar_width)

# Set the labels for the first y-axis (ax1)
ax1.set_xlabel('Fiscal Year', fontsize=12, fontweight='bold')
ax1.set_ylabel('FDI Inflow (US$ Billions)', color=bar_color, fontsize=12, fontweight='bold')
ax1.tick_params(axis='y', labelcolor=bar_color)
ax1.set_ylim(0, max(fdi_inflow_usd_billion) * 1.2) # Add some padding to the top

# --- Line Chart (Share of Total FDI) ---
# Create a second y-axis that shares the same x-axis
ax2 = ax1.twinx()
line_color = 'darkorange'
ax2.plot(years, share_of_total_fdi_percent, color=line_color, marker='o', linestyle='-', linewidth=2.5, label="Renewables' Share of Total FDI (%)")

# Set the labels for the second y-axis (ax2)
ax2.set_ylabel("Renewables' Share of Total FDI (%)", color=line_color, fontsize=12, fontweight='bold')
ax2.tick_params(axis='y', labelcolor=line_color)
ax2.set_ylim(0, max(share_of_total_fdi_percent) * 1.2) # Add padding

# --- Titles and Legends ---
# Main title for the graph
plt.title("FDI in India's Renewable Energy Sector (FY21-FY25)", fontsize=16, fontweight='bold', pad=20)

# To combine legends from both axes into one box
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines + lines2, labels + labels2, loc='upper left')

# Add data labels on top of the bars
for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2.0, yval + 0.1, f'${yval:.1f}B', ha='center', va='bottom')

# Ensure the layout is tight and clean
fig.tight_layout()

# --- Display the Graph ---
# Save the figure to a file
plt.savefig('fdi_renewables_india_chart.png', dpi=300)

# Show the plot
plt.show()